# SST DRVI factors — exploration

What does each factor *mean*? Whether the fit is trustworthy is settled in `01_qc.ipynb`;
this notebook assumes it is and goes after interpretation: per-factor activity, the gene
programs each direction encodes, and which factors are only half informative.

DRVI only — scVI's dimensions are not claimed to be individually interpretable, so there
are no traversal scores to read for it.

CPU-only. Kernel: `sst-drvi`.

In [ ]:
import json
import warnings

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

import drvi
import sst_drvi as S

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.set_figure_params(dpi=100, frameon=False, figsize=(4, 4))
pd.set_option("display.max_rows", 120)

## The fit being interpreted

Defaults to the covariate fit, falling back to the baseline if it is not on disk yet.
Switch `RUN` to compare.

In [ ]:
RUN, MODEL = "sst_k64_covar", "drvi"

model_path, embed_path = S.run_paths(RUN, MODEL)
if not embed_path.exists():
    RUN = "sst_k64"
    model_path, embed_path = S.run_paths(RUN, MODEL)
    print(f"covariate fit not on disk; falling back to {RUN}")

embed = ad.read_h5ad(embed_path)
gene_names = np.asarray(embed.uns["gene_names"]).astype(str)

print(f"run: {RUN}\n{embed}")
print("\nconfig:", json.dumps(json.loads(embed.uns["drvi_config"]), indent=2))
print("\nrun:   ", dict(embed.uns["drvi_run"]))
print("timings:", dict(embed.uns["timings"]))

## 1. Per-factor activity on the UMAP

One panel per non-vanished factor, ordered by reconstruction effect. Factors that light up
a single compact region are the interpretable ones.

In [ ]:
drvi.utils.pl.plot_latent_dims_in_umap(embed, remove_vanished=True, max_cells_to_plot=50_000)

## 2. Factor x group heatmaps

Mean factor value per group, on a balanced subsample so that large supertypes and regions
do not dominate.

In [ ]:
drvi.utils.pl.plot_latent_dims_in_heatmap(embed, "Supertype", remove_vanished=True)

In [ ]:
drvi.utils.pl.plot_latent_dims_in_heatmap(embed, "Brain Region", remove_vanished=True)

## 3. What genes does each factor encode?

The runner stored the out-of-distribution traversal scores in `embed.varm`;
`S.interpretability_scores` rebuilds the genes x factors table from them, so no model
reload (and no GPU) is needed here. `OOD_combined` is DRVI's recommended default;
`IND_max` scores the same factors within the observed data.

In [ ]:
scores = S.interpretability_scores(embed, gene_names, key="OOD_combined", directional=True)
print(f"{scores.shape[0]} genes x {scores.shape[1]} factor directions")
scores.iloc[:5, :8]

In [ ]:
top = S.top_genes_per_factor(scores, n_top=10)
top.pivot(index="rank", columns="factor", values="gene").iloc[:, :12]

In [ ]:
drvi.utils.pl.plot_interpretability_scores(scores, n_top_genes=10, ncols=4, score_threshold=0.1)

### Expression of the top genes of a chosen factor

Swap `factor` for anything from `scores.columns` to sanity-check that its top genes really do
vary with the factor. Counts are read from the prepared cohort for a random 20k-cell subsample
and normalized on the fly (the cohort stores raw counts only).

In [ ]:
factor = scores.columns[0]
factor_genes = scores[factor].nlargest(8).index.tolist()
base_title = factor.rstrip("+-")
dim = embed.var.index[embed.var["title"] == base_title][0]
print(factor, "->", factor_genes)

rng = np.random.default_rng(0)
sub_idx = np.sort(rng.choice(embed.n_obs, size=min(20_000, embed.n_obs), replace=False))

# The cohort stores raw counts only, so normalize the subsample here.
prepared = S.load_prepared(backed="r")
sub = prepared[embed.obs_names[sub_idx]].to_memory()[:, factor_genes].copy()
sc.pp.normalize_total(sub, target_sum=1e4)
sc.pp.log1p(sub)

panel = embed[sub_idx].copy()
panel.obs["factor"] = np.asarray(panel[:, dim].X).ravel()
for gene in factor_genes:
    panel.obs[gene] = np.asarray(sub[:, gene].X.todense()).ravel()

sc.pl.umap(panel, color=["factor"] + factor_genes, ncols=3, size=6, cmap="viridis")

## 4. Split each factor by sign, then re-test

A DRVI factor is directional — `DR n+` and `DR n-` are decoded as separate programs, and
`set_latent_dimension_stats` even flags them vanished independently. Associating on the
*signed* value therefore mixes two programs into one number: if a covariate drives only
the negative direction, the cells sitting above zero dilute it.

So split each factor into `relu(x)` and `relu(-x)` — two non-negative magnitude columns —
and re-run the same association test. Column labels match section 3's score table, so the
two can be read side by side.

In [ ]:
split = S.split_by_sign(embed)
print(f"{split.shape[1]} live directions from {embed.n_vars} factors")

activity = S.direction_activity(embed)
print("\nHow often is each direction active? (median frac_active tells you whether "
      "factors are genuinely two-sided)")
print(activity["frac_active"].describe()[["min", "25%", "50%", "75%", "max"]].round(3).to_string())
activity.head(10).round(3)

The signed association is also shown in `01_qc.ipynb`; it is recomputed here
because the comparison below needs both halves in one place.

In [ ]:
CATEGORICAL = ("Supertype", "Brain Region", "Donor ID", "library_prep", "method",
               "Chemistry", "Sex", "Cognitive Status", "Overall AD neuropathological Change",
               "Braak")
CONTINUOUS = ("CPS_Local", "CPS_Global", "CPS_Local_pTau", "CPS_Local_ABeta",
              "Age at Death", "PMI", "Number of UMIs", "Genes detected",
              "Fraction mitochondrial UMIs", "Doublet score")

assoc = S.factor_association(embed, categorical=CATEGORICAL, continuous=CONTINUOUS)
assoc_dir = S.factor_association(embed, categorical=CATEGORICAL, continuous=CONTINUOUS,
                                 directional=True)
assoc_dir.round(3)

### Did splitting help?

Per covariate, the strongest association before and after. Symmetric, monotone effects lose a
little (relu throws away half the variance); anything carried by one direction gains.

In [ ]:
comparison = pd.DataFrame({
    "signed_max": assoc.max().round(3),
    "sign_split_max": assoc_dir.max().round(3),
})
comparison["delta"] = (comparison["sign_split_max"] - comparison["signed_max"]).round(3)
comparison["best_direction"] = assoc_dir.idxmax()
comparison.sort_values("sign_split_max", ascending=False)

### Which factors are one-sided?

The gap `|assoc(+) - assoc(-)|` per factor. A large gap means the two halves of one latent
dimension track different things — so the factor is only half informative, and downstream
work should use the split columns rather than the raw signed value.

In [ ]:
asym = S.direction_asymmetry(assoc_dir)
worst = asym.max(axis=1).sort_values(ascending=False)
print("largest +/- disagreement per factor:")
print(worst.head(12).round(3).to_string())

In [ ]:
# Drill into the most asymmetric factors: what drives each side?
for f in worst.head(6).index:
    sides = pd.DataFrame({"+": assoc_dir.loc[f"{f}+"], "-": assoc_dir.loc[f"{f}-"]})
    sides["gap"] = sides["+"] - sides["-"]
    top4 = sides.reindex(sides["gap"].abs().sort_values(ascending=False).index).head(4)
    print(f"\n{f}:")
    print(top4.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(0.5 * assoc_dir.shape[1] + 3, 0.16 * assoc_dir.shape[0] + 2))
sns.heatmap(assoc_dir, cmap="rocket_r", vmin=0, vmax=1, linewidths=0.2,
            cbar_kws={"label": "eta$^2$ / |Spearman $\\rho$|"}, ax=ax)
ax.set_xlabel("")
ax.set_ylabel("factor direction")
ax.set_title("Factor-covariate association, split by sign")
fig.tight_layout()

## 5. Summary

Fill in after looking at the plots above:

- Factors tracking supertype identity:
- Factors tracking brain region:
- Factors tracking neuropathology (`CPS_*`, Braak, AD change):
- Factors that still look technical despite the covariate model:
- One-sided factors from section 4, where only `+` or only `-` is usable:

In [ ]:
from session_info2 import session_info

session_info(dependencies=True)